In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

print(f"Pandas  : {pd.__version__}")
print(f"Numpy   : {np.__version__}")
print("✅ Ready")

Pandas  : 3.0.3
Numpy   : 2.5.0
✅ Ready


In [2]:
users        = pd.read_csv('../data/raw/users.csv')
courses      = pd.read_csv('../data/raw/courses.csv')
transactions = pd.read_csv('../data/raw/transactions.csv')

# Fix date type immediately
transactions['TransactionDate'] = pd.to_datetime(transactions['TransactionDate'])

print("Data loaded:")
print(f"  Users        : {users.shape}")
print(f"  Courses      : {courses.shape}")
print(f"  Transactions : {transactions.shape}")

Data loaded:
  Users        : (2000, 3)
  Courses      : (400, 5)
  Transactions : (23141, 4)


In [3]:
# Join transactions with course info and user info
master = (transactions
          .merge(courses, on='CourseID', how='left')
          .merge(users,   on='UserID',   how='left'))

print(f"Master table shape : {master.shape}")
print(f"Columns            : {master.columns.tolist()}")
print()
master.head(3)

Master table shape : (23141, 10)
Columns            : ['UserID', 'CourseID', 'TransactionDate', 'Amount', 'CourseCategory', 'CourseType', 'CourseLevel', 'CourseRating', 'Age', 'Gender']



,UserID,CourseID,TransactionDate,Amount,CourseCategory,CourseType,CourseLevel,CourseRating,Age,Gender
0,U0001,C0131,2022-12-10,249.64,Data Science,Live,Beginner,4.5,56,Female
1,U0001,C0190,2025-07-26,132.82,Technology,Live,Intermediate,5.0,56,Female
2,U0001,C0073,2025-09-20,36.81,Health & Wellness,Hybrid,Beginner,3.6,56,Female


In [4]:
# One row per user — summary statistics
profiles = master.groupby('UserID').agg(

    # Enrollment behaviour
    total_courses      = ('CourseID',        'nunique'),  # unique courses only
    total_transactions = ('CourseID',        'count'),    # total purchases
    total_spending     = ('Amount',          'sum'),
    avg_spending       = ('Amount',          'mean'),
    min_spending       = ('Amount',          'min'),
    max_spending       = ('Amount',          'max'),

    # Course quality
    avg_course_rating  = ('CourseRating',    'mean'),

    # Category diversity
    n_categories       = ('CourseCategory',  'nunique'),

    # Time features
    first_enroll       = ('TransactionDate', 'min'),
    last_enroll        = ('TransactionDate', 'max'),

).reset_index()

print(f"Profiles shape : {profiles.shape}")
print(f"Columns        : {profiles.columns.tolist()}")
profiles.head(3)

Profiles shape : (2000, 11)
Columns        : ['UserID', 'total_courses', 'total_transactions', 'total_spending', 'avg_spending', 'min_spending', 'max_spending', 'avg_course_rating', 'n_categories', 'first_enroll', 'last_enroll']


,UserID,total_courses,total_transactions,total_spending,avg_spending,min_spending,max_spending,avg_course_rating,n_categories,first_enroll,last_enroll
0,U0001,19,19,3174.70,167.089474,36.81,278.97,4.273684,8,2022-04-19,2025-11-26
1,U0002,11,11,2356.88,214.261818,56.94,294.70,4.354545,7,2022-01-07,2025-05-23
2,U0003,12,12,2028.92,169.076667,37.46,281.29,4.058333,5,2022-03-10,2025-09-08


In [5]:
# Find which category each user enrolled in most
pref_cat = (master
            .groupby(['UserID', 'CourseCategory'])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
            .drop_duplicates(subset='UserID')
            [['UserID', 'CourseCategory']]
            .rename(columns={'CourseCategory': 'preferred_category'}))

# Merge into profiles
profiles = profiles.merge(pref_cat, on='UserID', how='left')

print("Preferred category added ✅")
print(profiles['preferred_category'].value_counts())

Preferred category added ✅
preferred_category
Technology           688
Data Science         356
Business             312
Design               203
Marketing            167
Finance              106
Health & Wellness    100
Arts & Creativity     68
Name: count, dtype: int64


In [6]:
# Find which level each user enrolled in most
pref_lvl = (master
            .groupby(['UserID', 'CourseLevel'])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
            .drop_duplicates(subset='UserID')
            [['UserID', 'CourseLevel']]
            .rename(columns={'CourseLevel': 'preferred_level'}))

# Merge into profiles
profiles = profiles.merge(pref_lvl, on='UserID', how='left')

print("Preferred level added ✅")
print(profiles['preferred_level'].value_counts())

Preferred level added ✅
preferred_level
Beginner        1225
Intermediate     694
Advanced          81
Name: count, dtype: int64


In [7]:
# Count how many courses each user took at each level
level_counts = (master
                .groupby(['UserID', 'CourseLevel'])
                .size()
                .unstack(fill_value=0)
                .reset_index())

# Make sure all 3 columns exist even if someone has 0 in one
for lvl in ['Beginner', 'Intermediate', 'Advanced']:
    if lvl not in level_counts.columns:
        level_counts[lvl] = 0

# Merge into profiles
profiles = profiles.merge(
    level_counts[['UserID', 'Beginner', 'Intermediate', 'Advanced']],
    on='UserID', how='left'
)

print("Level counts added ✅")
profiles[['UserID','Beginner','Intermediate','Advanced']].head(5)

Level counts added ✅


,UserID,Beginner,Intermediate,Advanced
0,U0001,8,8,3
1,U0002,5,4,2
2,U0003,6,5,1
3,U0004,6,3,4
4,U0005,4,7,3


In [8]:
# Bring in Age and Gender from users table
profiles = profiles.merge(
    users[['UserID', 'Age', 'Gender']],
    on='UserID', how='left'
)

print("Demographics added ✅")
profiles[['UserID','Age','Gender']].head(5)

Demographics added ✅


,UserID,Age,Gender
0,U0001,56,Female
1,U0002,46,Male
2,U0003,32,Male
3,U0004,60,Female
4,U0005,25,Male


In [9]:
# ── Time Features ──────────────────────────────
duration_days = (
    (profiles['last_enroll'] - profiles['first_enroll']).dt.days + 1
)

# How frequently does the user enroll (courses per active day)
profiles['enrollment_frequency'] = (
    profiles['total_courses'] / duration_days
)

# Days since last enrollment (recency)
reference_date = pd.Timestamp('2026-01-01')
profiles['recency_days'] = (
    reference_date - profiles['last_enroll']
).dt.days


# ── Diversity Features ─────────────────────────
# How many different categories explored (raw count)
profiles['diversity_score'] = profiles['n_categories']

# Category concentration (1 = only 1 category, 0 = all categories)
profiles['category_concentration'] = (
    1 - (profiles['n_categories'] / 8)
)


# ── Learning Depth Features ────────────────────
total_lvl = (
    profiles['Beginner'] +
    profiles['Intermediate'] +
    profiles['Advanced']
).replace(0, 1)  # avoid divide by zero

# Depth index: 0 = all beginner, 1 = all advanced
profiles['learning_depth_index'] = (
    (profiles['Intermediate'] + 2 * profiles['Advanced']) /
    (2 * total_lvl)
)

# What % of their courses are beginner level
profiles['beginner_ratio'] = profiles['Beginner'] / total_lvl

# What % of their courses are advanced level
profiles['advanced_ratio'] = profiles['Advanced'] / total_lvl


print("✅ All derived features calculated")
print()
print("New features added:")
new_feats = [
    'enrollment_frequency', 'recency_days',
    'diversity_score', 'category_concentration',
    'learning_depth_index', 'beginner_ratio', 'advanced_ratio'
]
for f in new_feats:
    print(f"  {f:<30} | mean={profiles[f].mean():.3f} | min={profiles[f].min():.3f} | max={profiles[f].max():.3f}")

✅ All derived features calculated

New features added:
  enrollment_frequency           | mean=0.035 | min=0.002 | max=1.000
  recency_days                   | mean=150.517 | min=2.000 | max=1423.000
  diversity_score                | mean=5.724 | min=1.000 | max=8.000
  category_concentration         | mean=0.284 | min=0.000 | max=0.875
  learning_depth_index           | mean=0.350 | min=0.000 | max=1.000
  beginner_ratio                 | mean=0.473 | min=0.000 | max=1.000
  advanced_ratio                 | mean=0.172 | min=0.000 | max=1.000


In [12]:
from sklearn.preprocessing import LabelEncoder

CATEGORIES = [
    'Technology', 'Data Science', 'Business',
    'Design', 'Marketing', 'Finance',
    'Health & Wellness', 'Arts & Creativity'
]
LEVELS = ['Beginner', 'Intermediate', 'Advanced']

le_cat = LabelEncoder()
le_lvl = LabelEncoder()

le_cat.fit(CATEGORIES)
le_lvl.fit(LEVELS)

profiles['preferred_category_enc'] = le_cat.transform(
    profiles['preferred_category'].fillna('Technology')
)
profiles['preferred_level_enc'] = le_lvl.transform(
    profiles['preferred_level'].fillna('Beginner')
)

print("Encoding mapping — Category:")
for i, cat in enumerate(le_cat.classes_):
    print(f"  {cat:<25} → {i}")

print("\nEncoding mapping — Level:")
for i, lvl in enumerate(le_lvl.classes_):
    print(f"  {lvl:<15} → {i}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Fill any remaining nulls with 0
profiles = profiles.fillna(0)

# Replace infinity values
profiles = profiles.replace([np.inf, -np.inf], 0)

# Verify no nulls remain
null_check = profiles.isnull().sum()
print("Null check after cleaning:")
print(null_check[null_check > 0] if null_check.sum() > 0 else "  No nulls ✅")

In [ ]:
CLUSTERING_FEATURES = [
    'total_courses',
    'avg_spending',
    'avg_course_rating',
    'diversity_score',
    'learning_depth_index',
    'beginner_ratio',
    'enrollment_frequency',
    'category_concentration',
    'n_categories',
    'recency_days',
    'preferred_category_enc',
    'preferred_level_enc'
]

print("═══════════════════════════════════════════")
print("   FINAL FEATURE MATRIX SUMMARY")
print("═══════════════════════════════════════════")
print(f"Total users    : {len(profiles)}")
print(f"Total features : {len(CLUSTERING_FEATURES)}")
print()
print(f"{'Feature':<30} {'Mean':>8} {'Min':>8} {'Max':>8}")
print("─" * 58)
for f in CLUSTERING_FEATURES:
    print(f"{f:<30} {profiles[f].mean():>8.3f} {profiles[f].min():>8.3f} {profiles[f].max():>8.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

# Chart 1 — Total courses per user
axes[0,0].hist(profiles['total_courses'], bins=30,
               color='#4361ee', edgecolor='white')
axes[0,0].set_title('Courses per User')
axes[0,0].set_xlabel('Total Courses')
axes[0,0].set_ylabel('Number of Users')
axes[0,0].axvline(profiles['total_courses'].mean(),
                  color='red', linestyle='--',
                  label=f"Mean = {profiles['total_courses'].mean():.1f}")
axes[0,0].legend()

# Chart 2 — Total spending per user
axes[0,1].hist(profiles['total_spending'], bins=30,
               color='#f72585', edgecolor='white')
axes[0,1].set_title('Total Spending per User')
axes[0,1].set_xlabel('Spending ($)')
axes[0,1].set_ylabel('Number of Users')
axes[0,1].axvline(profiles['total_spending'].mean(),
                  color='blue', linestyle='--',
                  label=f"Mean = ${profiles['total_spending'].mean():.0f}")
axes[0,1].legend()

# Chart 3 — Learning depth index
axes[1,0].hist(profiles['learning_depth_index'], bins=25,
               color='#7209b7', edgecolor='white')
axes[1,0].set_title('Learning Depth Index')
axes[1,0].set_xlabel('Depth Index (0=Beginner, 1=Advanced)')
axes[1,0].set_ylabel('Number of Users')

# Chart 4 — Preferred category
cat_counts = profiles['preferred_category'].value_counts()
axes[1,1].barh(cat_counts.index, cat_counts.values,
               color=sns.color_palette('husl', len(cat_counts)))
axes[1,1].set_title('Preferred Category Distribution')
axes[1,1].set_xlabel('Number of Users')

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/07_feature_distributions.png', dpi=150)
plt.show()
print("Saved ✅")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

corr = profiles[CLUSTERING_FEATURES].corr().round(2)

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Correlation'}
)

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/08_correlation_matrix.png', dpi=150)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Avg spending by preferred category
cat_spend = (profiles.groupby('preferred_category')['total_spending']
                     .mean()
                     .sort_values(ascending=False)
                     .reset_index())

ax1.barh(cat_spend['preferred_category'],
         cat_spend['total_spending'],
         color=sns.color_palette('husl', len(cat_spend)))
ax1.set_title('Avg Total Spend by Preferred Category', fontweight='bold')
ax1.set_xlabel('Avg Total Spending ($)')

# Avg depth index by preferred level
lvl_depth = (profiles.groupby('preferred_level')['learning_depth_index']
                     .mean()
                     .reset_index())

colors = {'Beginner':'#4cc9f0', 'Intermediate':'#4361ee', 'Advanced':'#7209b7'}
bar_colors = [colors[l] for l in lvl_depth['preferred_level']]
ax2.bar(lvl_depth['preferred_level'],
        lvl_depth['learning_depth_index'],
        color=bar_colors, edgecolor='white')
ax2.set_title('Avg Depth Index by Preferred Level', fontweight='bold')
ax2.set_ylabel('Learning Depth Index')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/figures/09_category_level_analysis.png', dpi=150)
plt.show()
print("Saved ✅")
plt.show()
print("Saved ✅")

In [ ]:
# Make sure output folder exists
os.makedirs('../data/processed', exist_ok=True)

# Save full profiles
profiles.to_csv('../data/processed/master_profiles.csv', index=False)

# Save just the feature matrix (for clustering)
feature_matrix = profiles[['UserID'] + CLUSTERING_FEATURES]
feature_matrix.to_csv('../data/processed/feature_matrix.csv', index=False)

print("═══════════════════════════════════════════")
print("   DAY 2 COMPLETE — FILES SAVED")
print("═══════════════════════════════════════════")
print(f"  master_profiles.csv  → {len(profiles)} rows × {len(profiles.columns)} columns")
print(f"  feature_matrix.csv   → {len(feature_matrix)} rows × {len(feature_matrix.columns)} columns")
print()
print("Saved to data/processed/ ✅")
print()
